## California Housing KMDS Logging

This notebook classifies each observation with the KMDS natural-language observation utility before logging it to the project knowledge graph.


In [3]:
import subprocess
import sys
from pathlib import Path

repo = Path("/home/rajiv/programming/KMDS")
try:
    import kmds
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(repo)])
    for candidate in [
        repo / "src",
        Path("/home/rajiv/programming/kmds-data-helper/src"),
    ]:
        if candidate.exists():
            sys.path.insert(0, str(candidate))
    import kmds

from kmds.utils.natural_language_observation import map_text_to_observation

PROJECT_FILE = Path(r"/home/rajiv/programming/kmds_migration/california_housing/project_knowledge_graph.xml")
WORKFLOW_NAME = "california_housing_workflow"

provisional_observations = [
    "The California Housing dataset was prepared with the kmds-dataset-util workflow and includes 20,640 rows across 9 numeric attributes for benchmarking.",
    "Data quality review: the kmds-dataset-util reports confirm the California Housing metadata is correct and that the cleaned schema matches the prepared dataset.",
    "Modeling choice: use mean squared error (MSE) as the objective, create train/validation/test splits, and tune the model on validation performance so the training and validation errors remain close.",
    "Model selection: the gradient boosting regressor was selected because it achieved the lowest validation MSE of 0.2610 and the best test MSE of 0.2299, yielding a final test R² of 0.8261.",
]

for i, text in enumerate(provisional_observations, start=1):
    mapping = map_text_to_observation(text)
    print(f"Provisional observation {i}: {mapping.observation_type}, valid={mapping.validation_passed}")


Provisional observation 1: Model Selection Setup Description, valid=True
Provisional observation 2: Data Quality Observation, valid=True
Provisional observation 3: Data Quality Observation, valid=True
Provisional observation 4: Model Selection Statement, valid=True


In [4]:
corrected_observations = [
    "Data quality review: The California Housing dataset was prepared with the kmds-dataset-util workflow and includes 20,640 rows across 9 numeric attributes for benchmarking.",
    "Data quality review: the kmds-dataset-util reports confirm the California Housing metadata is correct and that the cleaned schema matches the prepared dataset.",
    "Modeling choice: use mean squared error (MSE) as the objective, create train/validation/test splits, and tune the model on validation performance so the training and validation errors remain close.",
    "Model selection: the gradient boosting regressor was selected because it achieved the lowest validation MSE of 0.2610 and the best test MSE of 0.2299, yielding a final test R² of 0.8261.",
]

print("Correction: the provisional sentence 'The California Housing dataset was prepared ...' was misclassified as model selection. Under the KMDS glossary, this is an Exploratory Observation and specifically a Data Quality Observation.")
print("The corrected and valid form used below is: 'Data quality review: The California Housing dataset was prepared with the kmds-dataset-util workflow and includes 20,640 rows across 9 numeric attributes for benchmarking.'")

for i, text in enumerate(corrected_observations, start=1):
    mapping = map_text_to_observation(text)
    print(f"Corrected observation {i}: {mapping.observation_type}, valid={mapping.validation_passed}")


Correction: the provisional sentence 'The California Housing dataset was prepared ...' was misclassified as model selection. Under the KMDS glossary, this is an Exploratory Observation and specifically a Data Quality Observation.
The corrected and valid form used below is: 'Data quality review: The California Housing dataset was prepared with the kmds-dataset-util workflow and includes 20,640 rows across 9 numeric attributes for benchmarking.'
Corrected observation 1: Data Quality Observation, valid=True
Corrected observation 2: Data Quality Observation, valid=True
Corrected observation 3: Data Quality Observation, valid=True
Corrected observation 4: Model Selection Statement, valid=True


In [5]:
from kmds.utils.natural_language_observation import log_text_as_observation

for i, text in enumerate(corrected_observations, start=1):
    mapping = map_text_to_observation(text)
    if not mapping.validation_passed:
        raise ValueError(f"Invalid corrected observation {i}: {mapping.validation_errors}")
    result = log_text_as_observation(
        text=text,
        workflow_name=WORKFLOW_NAME,
        project_file_path=str(PROJECT_FILE),
        project_mode="update",
        workflow_type="application",
        finding_sequence=i,
    )
    print(f"Logged corrected observation {i}: {result.mapping.observation_type}")


Logged corrected observation 1: Data Quality Observation
Logged corrected observation 2: Data Quality Observation
Logged corrected observation 3: Data Quality Observation
Logged corrected observation 4: Model Selection Statement
